In [22]:
import numpy as np
from scipy.special import softmax

# 1. 단어 벡터 생성
words = {
    "나는" : np.array([1.0, 0.0]),
    "사과를" : np.array([1.0, 1.0]),
    "먹는다" : np.array([0.0, 1.0]),
}

# 2. 문장을 행렬로 변경
X = np.array(list(words.values()))
print(f"\nX=\n{X}")

# Q : 찾고 싶은 정보, K : 다른 단어들의 특징, V : 실제 전달할 내용
Q = X
K = X
V = X

print(f"\nQ=\n{Q}")
print(f"\nK=\n{K.T}")
print(f"\nQ*K=\n{np.dot(Q, K.T)}")

attention_weights = softmax(np.dot(Q, K.T), axis=1)
print(f"\nattention_weights=\n{attention_weights}")

# 단어 자체의 의미를 주변 문맥을 반영한 의미로 변환됨
output = np.dot(attention_weights, V)
print(f"\noutput=\n{output}")


X=
[[1. 0.]
 [1. 1.]
 [0. 1.]]

Q=
[[1. 0.]
 [1. 1.]
 [0. 1.]]

K=
[[1. 1. 0.]
 [0. 1. 1.]]

Q*K=
[[1. 1. 0.]
 [1. 2. 1.]
 [0. 1. 1.]]

attention_weights=
[[0.4223188  0.4223188  0.1553624 ]
 [0.21194156 0.57611688 0.21194156]
 [0.1553624  0.4223188  0.4223188 ]]

output=
[[0.8446376  0.5776812 ]
 [0.78805844 0.78805844]
 [0.5776812  0.8446376 ]]


In [1]:
# 토크나이즈 허깅페이스에서 다운로드하기
from transformers import AutoTokenizer

model_name = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [24]:
# 문장을 단어로 구분하기
text = "나는 사과를 좋아한다"

tokenizer.tokenize(text)

['나', '##는', '사과', '##를', '좋아한다']

In [25]:
# 문장를 숫자로 변경
ids = tokenizer.encode(text)
ids

[2, 717, 2259, 5180, 2138, 17331, 3]

In [26]:
vocab_size = tokenizer.vocab_size
vocab_size

32000

In [27]:
tokenizer.model_max_length

512

In [35]:
texts = [
    "오늘 그만 지각을 해버렸네요",
    "맑은 날씨를 좋아합니다.",
    "기분이 안좋네요"
]

inputs = tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors="np"
)

# input_ids, token_type_ids, attention_mask의 키로 구성됨
inputs['input_ids']

array([[    2,  3822,  4416,  8955,  2069,  1897, 15763,  2203,  2182,
            3],
       [    2,  1042,  2073,  5792,  2138,  5723, 11800,    18,     3,
            0],
       [    2,  4754,  2052,  1378,  2716,  2203,  2182,     3,     0,
            0]])

In [33]:
import tensorflow as tf
input_ids = tf.convert_to_tensor(inputs["input_ids"])
attention_mask = tf.convert_to_tensor(inputs["attention_mask"])

input_ids, attention_mask

(<tf.Tensor: shape=(3, 10), dtype=int64, numpy=
 array([[    2,  3822,  4416,  8955,  2069,  1897, 15763,  2203,  2182,
             3],
        [    2,  1042,  2073,  5792,  2138,  5723, 11800,    18,     3,
             0],
        [    2,  4754,  2052,  1378,  2716,  2203,  2182,     3,     0,
             0]])>,
 <tf.Tensor: shape=(3, 10), dtype=int64, numpy=
 array([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0]])>)

In [2]:
# 감정분석 => 영화리뷰를 읽어서 긍정/부정인지 판별
import pandas as pd

train = pd.read_csv("http://114.207.245.181:13000/txt/ratings_train.txt", sep="\t")
test = pd.read_csv("http://114.207.245.181:13000/txt/ratings_test.txt", sep="\t")